# 00.- Cargar fichero

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
from datetime import datetime
import os


In [26]:
def importar_csv_df(fichero:str)->pd.DataFrame:
    """Recibe el nombre de un fichero csv de los datos OHLC con su path si esta fuera del directorio de trabajo 
    para cargarlo y devolverlo en un DataFramecon las columnas: "date", "open", "high", "low", "close" y "volume"

    Args:
        fichero (str): Nombre del fichero csv a cargar en el DF        

    Returns:
        pd.DataFrame: DF con con el fichero csv cargado
    """
    
    columnas = ["date", "open", "high", "low", "close", "volume"]
    
    try:
        df = pd.read_csv(fichero, sep=",", 
                skiprows=[0,1,2], 
                names=columnas, 
                parse_dates=["date"])  # para que date sea del tipo datetime
        return(df)
    
    except FileNotFoundError:
        print(f"ERROR: no se encontró el fichero '{fichero}'")
        return None

# 01.- Comprobar y limpiar

In [3]:
def comprobar_non_null(df:pd.DataFrame)->bool:
    """Comprueba si todas las columnas de un DataFrame tienen el mismo número de valores no nulos.

    Calcula el número de valores no nulos por columna y verifica si ese recuento
    es idéntico en todas ellas. Si todas las columnas tienen la misma cantidad de
    valores no nulos, devuelve True; en caso contrario, devuelve False.

    Args:
        df (pd.DataFrame): DataFrame sobre el que se realiza la comprobación.

    Returns:
        bool: True si todas las columnas tienen el mismo número de valores no nulos, False en caso contrario.
    """
    non_null_counts = df.notnull().sum()
    todos_iguales = non_null_counts.nunique() == 1

    if todos_iguales:
        valor = non_null_counts.iloc[0] # todas las columnas tienen este valor
    else:
        max_count = non_null_counts.max()
    
    return todos_iguales

# 02.- Crear calendario completo

In [4]:
# Vamos a crear un calendario completo del 1 de enero al 31 de diciembre para cada año, 
# y luego insertar esos días en el DataFrame rellenando con NaN para que no se tengan en cuenta al hacer las operaciones estadisticas

def crea_calendario_completo(df:pd.DataFrame)->pd.DataFrame:
    """Genera un DataFrame con un calendario completo para cada año presente en la columna 'date',
    insertando filas para todos los días del año y rellenando con NaN los días que no existían
    en el DataFrame original.

    El objetivo es disponer de un calendario continuo (1 de enero a 31 de diciembre) para cada año,
    de modo que las operaciones estadísticas posteriores no se vean afectadas por la ausencia de días.
    Los días añadidos se rellenan con NaN para que no influyan en medias, desviaciones u otros cálculos.

    Args:
        df (pd.DataFrame): DataFrame original que contiene al menos una columna 'date' de tipo datetime.

    Returns:
        pd.DataFrame: Nuevo DataFrame con:
        - Todas las fechas del calendario completo para cada año detectado.
        - Las columnas originales del DataFrame, con NaN en los días añadidos.
        - Una columna adicional 'weekday' con el nombre del día de la semana.
    """
    # 1. Obtener lista de años presentes en el DataFrame
    years = df["date"].dt.year.unique()

    # 2. Crear un índice de fechas vacío donde iremos añadiendo los días completos de cada año.
    rango_completo = pd.DatetimeIndex([])

    for y in years:
        inicio = f"{y}-01-01"
        fin = f"{y}-12-31"
        rango_completo = rango_completo.union(pd.date_range(inicio, fin, freq="D"))

    # 3. Reindexar para añadir días faltantes
    df = df.set_index("date")
    # Inserta todas las fechas del calendario completo
    # Si una fecha no existía en el DataFrame original → se crea una fila nueva

    df = df.reindex(rango_completo)

    # 4. Restaurar columna Date
    df.index.name = "date"
    df = df.reset_index()

    # 5. Añado columna con el dia de la semana
    df["weekday"] = df["date"].dt.day_name()
    
    return(df)

# 05.- Calculo primer dia del año operativo

In [5]:
# Calculo la primera fecha de cada año en la que su "close" no es NaN para ser el  primer dia del año con rendimiento (%) = 0

def calcular_dia_0_por_year(df:pd.DataFrame)->pd.DataFrame:
    """Identifica, para cada año presente en el DataFrame, la primera fecha en la que
    la columna 'close' tiene un valor no nulo, y construye un nuevo DataFrame con
    esa información.

    Esta función permite determinar el "día 0" de cada año, entendido como la
    primera fecha válida para calcular rendimientos anuales. Si un año no tiene
    ningún valor válido en 'close', se muestra un aviso por consola.

    Args:
        df (pd.DataFrame): DataFrame que debe contener al menos las columnas:
        - 'date' : fechas en formato datetime
        - 'year' : año correspondiente a cada fila
        - 'close': valor numérico que puede contener NaN

    Returns:
        pd.DataFrame: DataFrame con tres columnas:
        - 'year'     : año analizado
        - 'fecha_0'  : primera fecha del año con 'close' no nulo
        - 'close'    : valor de 'close' en esa fecha

        Si un año no tiene valores válidos, no se incluye en el resultado.
    """

    years = df["date"].dt.year.unique()
    # Crear el nuevo DataFrame vacío que tendrá el primer valor de close distinto de NaN y su fecha para cada año
    df_dia_0 = pd.DataFrame(columns=["year", "fecha_0", "close"])

    for y in years:
        df_year = df[df["year"] == y]          # Filtrar solo ese año
        df_valid = df_year.dropna(subset=["close"])  # Quitar filas donde close es NaN
        
        if len(df_valid) > 0:
            primera_fila = df_valid.iloc[0]    # Primera fila válida
            
            # Añadir una fila al nuevo DataFrame 
            df_dia_0.loc[len(df_dia_0)] = { "year": y, 
                                        "fecha_0": primera_fila["date"], 
                                        "close": primera_fila["close"] 
                                        }
                    
        else:
            print(f"Año {y}: no hay valores válidos")
            
    return df_dia_0

# 06.- Rendimiento acumulado por día

In [6]:
def calculo_rend_acum_dia(df:pd.DataFrame, df_dia_0:pd.DataFrame)->pd.DataFrame:
    """Calcula el rendimiento acumulado diario para cada año tomando como referencia
    el primer día válido ('día 0') de ese año, cuyo precio inicial se obtiene del
    DataFrame df_dia_0.

    Para cada año presente en el DataFrame original, se identifica el precio inicial
    correspondiente al primer día con valor válido (previamente calculado en df_dia_0),
    y se calcula el rendimiento acumulado diario mediante:

        rendimiento = (close / precio_inicial) - 1

    El resultado se almacena en una nueva columna llamada 'rendimiento_acumulado'en formato decimal, no en %.

    Args:
        df (pd.DataFrame): DataFrame con las columnas:
        - 'date'  : fechas en formato datetime
        - 'year'  : año correspondiente a cada fila
        - 'close' : precio de cierre diario (puede contener NaN)
        df_dia_0 (pd.DataFrame): DataFrame con las columnas:
        - 'year'     : año
        - 'fecha_0'  : primera fecha válida del año
        - 'close'    : precio inicial del año (día 0)

        Este DataFrame debe contener exactamente un registro por año.

    Returns:
        pd.DataFrame: El DataFrame original con una columna adicional:
        - 'rendimiento_acumulado' : rendimiento acumulado diario respecto al día 0 en formato decimal, no en %.
    """
    df = df.sort_values(by='date', ascending=True)
    
    years = df["date"].dt.year.unique()

    df["rendimiento_acumulado"] = np.nan  # columna vacía

    for y in years:
        precio_inicial = df_dia_0.loc[df_dia_0["year"] == y, "close"].iloc[0]
        mask = df["year"] == y
        df.loc[mask, "rendimiento_acumulado"] = ((df.loc[mask, "close"] / precio_inicial - 1).round(4))
    
    return df

# 10.- Rellenar NaN

In [ ]:

def rellena_nan(df_train:pd.DataFrame)->pd.DataFrame:
    """Rellena valores NaN en un DataFrame de rendimientos anuales siguiendo reglas
    específicas para garantizar que los cálculos estadísticos posteriores sean
    coherentes y no se vean afectados por huecos en los datos.

    La función realiza tres operaciones principales:
    1. Para cada columna (cada año), identifica el primer día con un valor válido
       y rellena con 0 todos los días anteriores, ya que antes del inicio real de
       cotización el rendimiento debe considerarse 0.
    2. En la columna 'rendimiento_medio', detecta el primer día cuyo valor es 0
       y establece a 0 todos los días anteriores, asegurando consistencia en el
       cálculo del rendimiento medio.
    3. Rellena todos los NaN restantes del DataFrame mediante forward-fill (ffill),
       propagando el último valor válido hacia adelante.

    Args:
        df_train (pd.DataFrame): DataFrame con índices de fecha y columnas correspondientes a años y a
        'rendimiento_medio'. Los valores pueden contener NaN antes del inicio
        real de cotización o en días sin datos.

    Returns:
        pd.DataFrame: El DataFrame original con:
        - NaN iniciales reemplazados por 0 antes del primer día válido de cada año.
        - 'rendimiento_medio' ajustado para que todos los días previos al primer 0
          también sean 0.
        - Resto de NaN rellenados mediante forward-fill.
    """
    # Rellenar los NaN anteriores al primer dia de cotizacion
    for year in df_train.columns:
    

        # Serie del año (columna entera de un año)
        serie = df_train[year]

        # Primer día con cotización real (con dato distinto a NaN)
        primer_dia = serie.first_valid_index()

        # Si el año tiene algún día válido
        if primer_dia is not None:
            # Días anteriores al primer día
            dias_previos = serie.loc[:primer_dia].index[:-1]

            # Poner esos NaN a 0
            df_train.loc[dias_previos, year] = 0

    # Detectar el primer día con valor 0 en rendimiento_medio
    primer_cero = df_train[df_train["rendimiento_medio"] == 0].index[0]

    # Poner a cero todos los días anteriores
    df_train.loc[:primer_cero, "rendimiento_medio"] = 0


    # Rellenar los NaN restantes de todo el DF con el último valor valido anterior
    df_train = df_train.ffill()

    return df_train

# 11.- Columna md

In [9]:
def columna_md(df:pd.DataFrame)->pd.DataFrame:
    """Añade una columna con el formato mes-día ('md') a partir de la columna 'date'
    y ordena el DataFrame cronológicamente dentro de cada año según ese formato.

    La columna 'md' permite comparar o visualizar datos por estacionalidad,
    independientemente del año, ya que transforma la fecha en un formato MM-DD.
    Tras crearla, el DataFrame se ordena primero por 'md' (enero → diciembre)
    y luego por 'year' para mantener la secuencia temporal dentro de cada año.

    Args:
        df (pd.DataFrame): DataFrame que debe contener al menos:
        - 'date' : columna datetime
        - 'year' : año correspondiente a cada fila

    Returns:
        pd.DataFrame: El DataFrame original con:
        - Nueva columna 'md' en formato MM-DD.
        - Filas ordenadas por 'md' y luego por 'year'.
    """
    # Crear columna mes-día (formato MM-DD)
    df["md"] = df["date"].dt.strftime("%m-%d")

    # Ordenar por mes-día para que el eje X vaya de enero a diciembre
    df = df.sort_values(["md", "year"])
    
    return df

# 100.- Cargar y limpiar

In [11]:
# ********************* PRINCIPAL 1 ***********************************************************************

def carga_limpia(name:str, ticker:str, output_folder:str)->tuple:
    """Carga un fichero CSV con datos OHLC de una acción, realiza comprobaciones de calidad
    y devuelve un DataFrame limpio junto con la lista de años presentes en los datos.

    El proceso incluye:
    - Carga del fichero CSV correspondiente a la acción.
    - Comprobación de valores nulos en todas las columnas.
    - Detección de filas duplicadas.
    - Verificación de fechas repetidas.
    - Ordenación cronológica del DataFrame.
    - Identificación de días faltantes respecto al rango completo de fechas.
    - Creación de un calendario completo (incluyendo fines de semana y festivos),
      rellenando con NaN los días sin datos reales.
    - Añadido de la columna 'year' con el año de cada fecha.
    - Obtención de la lista de años presentes en el DataFrame.

    Args:
        name (str): Nombre de la empresa (por ejemplo, "Amazon").
        ticker (str): Ticker bursátil de la empresa (por ejemplo, "AMZN").
        output_folder (str): Carpeta donde se encuentra el fichero CSV a cargar.

    Returns:
        tuple (df, years): df : pd.DataFrame
            DataFrame limpio y ordenado, con calendario completo y columna 'year'.
        years : np.ndarray
            Array con los años presentes en los datos.
    """
    
    # 0.- Cargar el fichero csv con lso valores OHLC de una acción
    fichero_carga = (f"{output_folder}/{name}_{ticker}.csv")

    # fichero_carga = r"C:\IG\BCN_Activa\d_Especializacion\sprint_13\SP500_Top25_data\Amazon_AMZN.csv"

    df = importar_csv_df(fichero_carga)

    # 1.- Comprobar si hay valores nulos en alguna columna
    no_hay_nulos = comprobar_non_null(df)
    if no_hay_nulos:
        print(f"{name} ✔ No hay ninguna columna con valores nulos")
    else:
        print(f"{name} WARNING: hay columnas con valores nulos")
        display(df.isna().sum())


    # 2.- Comprobar si hay duplicados
    duplicados_fila = df.duplicated().sum()
    if duplicados_fila == 0:
        print(f"{name} ✔ No hay ninguna fila con valores duplicados")
    else:
        print(f"{name} WARNING: Hay filas con valores duplicados, Total nulos ={duplicados_fila}")
        

    # 3.- Fechas repetidas (True si no hay repetidas)
    if df["date"].is_unique:
        print(f"{name} ✔ No hay fechas repetidas")
    else:
        print(f"{name} WARNING: Hay fechas repetidas")


    # 5.- Ordenar el Df por fecha ascendente
    df = df.sort_values("date")

    # 6.- Comprobar los dias que faltan (Faltan sábados, domingos y festivos)
    rango_completo = pd.date_range(start=df["date"].min(), end=df["date"].max(), freq="D")
    faltantes = rango_completo.difference(df["date"])

    # 7.- Crear calendario con todos los dias del año (incluido el 29 de feberero para años visiestos)
    df = crea_calendario_completo(df)
    # ver qué días se añadieron
    df[df.isna().any(axis=1)]["date"]

    # 8.- Crear una columna con el año de la fecha
    df["year"] = df["date"].dt.year


    # 9.- definir una lista con todos los años
    years = df["date"].dt.year.unique()
    
    return df, years

# 200.- Hacer ficheros tratados

In [13]:
def hacer_ficheros_tratados(df:pd.DataFrame)->tuple:
    """Procesa un DataFrame con datos diarios de una acción para generar las estructuras
    necesarias para el análisis estacional y el backtest. El flujo incluye la creación
    de columnas auxiliares, el cálculo de rendimientos acumulados y la separación en
    conjuntos de entrenamiento y validación.

    El proceso realiza las siguientes operaciones:
    1. Añade la columna 'md' (mes-día en formato MM-DD) para facilitar análisis estacionales.
    2. Identifica el primer día válido de cada año ('día 0') y su precio inicial.
    3. Calcula el rendimiento acumulado diario de cada año respecto a su día 0.
    4. Genera una tabla pivotada donde:
       - El índice es 'md' (MM-DD).
       - Cada columna representa un año.
       - Los valores son los rendimientos acumulados.
    5. Divide el DataFrame pivotado en:
       - df_train: años 2000–2023.
       - df_validacion: años 2024 en adelante.
    6. Calcula el rendimiento medio por día del año dentro del conjunto de entrenamiento.

    Args:
        df (pd.DataFrame): DataFrame original con columnas:
        - 'date' : fecha en formato datetime
        - 'year' : año correspondiente a cada fila
        - 'close': precio de cierre diario
        Además puede contener otras columnas generadas previamente.

    Returns:
        tuple: (df, df_train, df_validacion)
        df : pd.DataFrame
            DataFrame original enriquecido con columnas adicionales como 'md' y
            'rendimiento_acumulado'.
        df_train : pd.DataFrame
            Tabla pivotada con rendimientos acumulados para los años 2000–2023,
            incluyendo la columna 'rendimiento_medio'.
        df_validacion : pd.DataFrame
            Tabla pivotada con rendimientos acumulados para los años 2024 en adelante.
    """
    # 0.- Poner columna md
    df = columna_md(df)

    # 3.- Calculo la primera fecha de cada año en la que su "close" no es NaN para ser el  primer dia del año con rendimiento (%) = 0
    df_dia_0 = calcular_dia_0_por_year(df)

    # 4.- Calcular el rendimiento acumulado estando los NaN
    df = calculo_rend_acum_dia(df, df_dia_0)

    # 6.- # Creo una tabla con el indice "md" y cada columna un año y los valores seran el rendimiento acumulado para ese dia en cada año
    df_rendimientos = df.pivot(index="md", columns="year", values="rendimiento_acumulado")

    # 7.- Separo en 2 DF uno para train del 2000 al 2023 y otro para validación con 2024, 2025 y lo que llevamos de 2026

    df_train = df_rendimientos.loc[:, df_rendimientos.columns <= 2023]
    df_validacion = df_rendimientos.loc[:, df_rendimientos.columns >= 2024]

    # 8.- Media por filas del precio de close para cada dia del año teniendo en cuenta que los NaN no se tienen en cuenta en el cálculo estadístico

    df_train = df_train.copy()
    df_train["rendimiento_medio"] = df_train.mean(axis=1).round(4)
    
    return df, df_train, df_validacion


# 300.- EXPLICACION metricas entre 2 fechas

Metricas

🟩 👉 La opción correcta es: mantener NaN y NO rellenarlos antes de calcular la media.

🟦 ✔ Razón 1 — Rellenar festivos distorsiona la media
 - Si rellenas un festivo con el valor del día anterior, estás diciendo:
    - “El rendimiento del 20 de mayo es igual al del 19 de mayo”.
 - Pero eso no es cierto: simplemente no hubo cotización.
 - Si haces esto, la media del 20 de mayo incluirá valores que no corresponden a ese día, sino al día anterior. Eso introduce un sesgo artificial.

🟩 ✔ Razón 2 — La media debe representar el comportamiento REAL del día
Si un año el 20 de mayo cae en sábado:

 - No hay cotización
 - No hay rendimiento
 - Ese año no debe influir en la media del 20 de mayo

Esto es exactamente lo que hacen:

 - Bloomberg
 - Reuters
 - Yahoo Finance
 - Cualquier motor de backtesting serio

🟩 ✔ Versión final del flujo:
1. Calcular rend_acum con NaN incluidos
2. Calcular la media por día del año (NaN ignorados)
3. Rellenar NaN para cálculos posteriores:
    - primeros días del año → 0
    - festivos → ffill
4. Calcular rendimientos entre dos fechas

🟩 ✔ Resumen final. El pipeline quedará así:
 - 1️⃣ Calcular rend_acum por año (con NaN incluidos)
 - 2️⃣ Calcular rendimiento_medio (NaN ignorados)
 - 3️⃣ Poner a 0 los días previos al primer día de cotización de cada año
 - 4️⃣ Poner a 0 esos mismos días en rendimiento_medio
 - 5️⃣ Rellenar el resto de NaN con ffill()
 - 6️⃣ Ya puedes calcular rendimientos entre fechas sin problemas

# 310.- Exportar ficheros tratados

In [ ]:
def exportar_ficheros_tratados(df:pd.DataFrame, df_train:pd.DataFrame,df_validacion:pd.DataFrame):
    """Exporta a disco los DataFrames procesados (df, df_train y df_validacion) tras
    realizar una limpieza adicional sobre el conjunto de entrenamiento. Esta función
    prepara los datos finales que se utilizarán en el análisis estacional y en el
    backtest.

    El proceso incluye:
    1. Limpieza de df_train mediante:
       - Sustitución de NaN anteriores al primer día válido de cada año por 0.
       - Forward-fill del resto de valores faltantes.
    2. Creación de la carpeta de salida si no existe.
    3. Exportación de los tres DataFrames a ficheros Parquet dentro de la carpeta
       'SP500_Top25_tratados', con nombres basados en el nombre de la acción y su ticker.

    Args:
        df (pd.DataFrame): DataFrame principal con datos diarios, incluyendo columnas como 'date',
            'year', 'md' y 'rendimiento_acumulado'.
        df_train (pd.DataFrame): DataFrame pivotado con rendimientos acumulados para los años de entrenamiento
            (2000–2023), que será limpiado antes de exportarse.
        df_validacion (pd.DataFrame): DataFrame pivotado con rendimientos acumulados para los años de validación
            (2024 en adelante).
    """


    # 1.- Rellenar los NaN anteriores al primer dia de cotizacion con cero y el resto con el valor anterior valido
    df_train = rellena_nan(df_train)


    # Guardar df, df_train y df_validacion
    output_folder_out = "SP500_Top25_tratados"

    try:
        os.makedirs(output_folder_out, exist_ok=True)
        
        df.to_parquet(f"{output_folder_out}/{name}_{ticker}_df.parquet", index=True)
        df_train.to_parquet(f"{output_folder_out}/{name}_{ticker}_train.parquet", index=True)
        df_validacion.to_parquet(f"{output_folder_out}/{name}_{ticker}_validacion.parquet", index=True)
        
    except Exception as e: 
        print(f"❌ Error inesperado guardando: {e}")

✔️ Comparativa guardar en "csv" o en "parquet"

| Característica        | CSV              | Parquet              |
|-----------------------|------------------|-----------------------|
| Tamaño                | grande           | pequeño               |
| Velocidad             | lenta            | rápida                |
| Tipos de datos        | se pierden       | se conservan          |
| Precisión             | se pierde        | se conserva           |
| Índice                | se pierde        | se conserva           |
| Igualdad exacta       | casi imposible   | sí                    |
| Compatibilidad        | universal        | análisis de datos     |


# 400.- principal final


In [ ]:
# ---------------------------------------------------------
# 1. Lista de las 25 mayores empresas del S&P 500 por market cap
#    (Tickers oficiales de Yahoo Finance)
# ---------------------------------------------------------
sp500_top25 = {
    "Apple": "AAPL",
    "Microsoft": "MSFT",
    "Nvidia": "NVDA",
    "Amazon": "AMZN",
    "Meta": "META",
    "Alphabet Class A": "GOOGL",
    "Alphabet Class C": "GOOG",
    "Berkshire Hathaway": "BRK-B",
    "Eli Lilly": "LLY",
    "Broadcom": "AVGO",
    "Tesla": "TSLA",
    "JPMorgan": "JPM",
    "UnitedHealth": "UNH",
    "Visa": "V",
    "Exxon Mobil": "XOM",
    "Mastercard": "MA",
    "Johnson & Johnson": "JNJ",
    "Procter & Gamble": "PG",
    "Home Depot": "HD",
    "Costco": "COST",
    "Merck": "MRK",
    "Chevron": "CVX",
    "AbbVie": "ABBV",
    "PepsiCo": "PEP",
    "Walmart": "WMT"
}


output_folder = "SP500_Top25_data"

for name, ticker in sp500_top25.items():

    # 1.- Ejecutar principal 1
    df, years = carga_limpia(name, ticker, output_folder)

    # 2.- Ejecutar principal 2
    df, df_train, df_validacion = hacer_ficheros_tratados(df)

    # 3.- Guardar los ficheros tratados en un directorio
    exportar_ficheros_tratados(df, df_train,df_validacion)

Apple ✔ No hay ninguna columna con valores nulos
Apple ✔ No hay ninguna fila con valores duplicados
Apple ✔ No hay fechas repetidas


c:\Users\Ignacio\anaconda3\envs\mt5python\Lib\site-packages\pandas\io\parquet.py:190: UserWarning: The DataFrame has column names of mixed type. They will be converted to strings and not roundtrip correctly.
  table = self.api.Table.from_pandas(df, **from_pandas_kwargs)
c:\Users\Ignacio\anaconda3\envs\mt5python\Lib\site-packages\pandas\io\parquet.py:190: UserWarning: The DataFrame has column names of mixed type. They will be converted to strings and not roundtrip correctly.
  table = self.api.Table.from_pandas(df, **from_pandas_kwargs)


Microsoft ✔ No hay ninguna columna con valores nulos
Microsoft ✔ No hay ninguna fila con valores duplicados
Microsoft ✔ No hay fechas repetidas
Nvidia ✔ No hay ninguna columna con valores nulos
Nvidia ✔ No hay ninguna fila con valores duplicados
Nvidia ✔ No hay fechas repetidas
Amazon ✔ No hay ninguna columna con valores nulos
Amazon ✔ No hay ninguna fila con valores duplicados
Amazon ✔ No hay fechas repetidas


c:\Users\Ignacio\anaconda3\envs\mt5python\Lib\site-packages\pandas\io\parquet.py:190: UserWarning: The DataFrame has column names of mixed type. They will be converted to strings and not roundtrip correctly.
  table = self.api.Table.from_pandas(df, **from_pandas_kwargs)
c:\Users\Ignacio\anaconda3\envs\mt5python\Lib\site-packages\pandas\io\parquet.py:190: UserWarning: The DataFrame has column names of mixed type. They will be converted to strings and not roundtrip correctly.
  table = self.api.Table.from_pandas(df, **from_pandas_kwargs)


Meta ✔ No hay ninguna columna con valores nulos
Meta ✔ No hay ninguna fila con valores duplicados
Meta ✔ No hay fechas repetidas
Alphabet Class A ✔ No hay ninguna columna con valores nulos
Alphabet Class A ✔ No hay ninguna fila con valores duplicados
Alphabet Class A ✔ No hay fechas repetidas


...

c:\Users\Ignacio\anaconda3\envs\mt5python\Lib\site-packages\pandas\io\parquet.py:190: UserWarning: The DataFrame has column names of mixed type. They will be converted to strings and not roundtrip correctly.
  table = self.api.Table.from_pandas(df, **from_pandas_kwargs)
c:\Users\Ignacio\anaconda3\envs\mt5python\Lib\site-packages\pandas\io\parquet.py:190: UserWarning: The DataFrame has column names of mixed type. They will be converted to strings and not roundtrip correctly.
  table = self.api.Table.from_pandas(df, **from_pandas_kwargs)


Merck ✔ No hay ninguna columna con valores nulos
Merck ✔ No hay ninguna fila con valores duplicados
Merck ✔ No hay fechas repetidas
Chevron ✔ No hay ninguna columna con valores nulos
Chevron ✔ No hay ninguna fila con valores duplicados
Chevron ✔ No hay fechas repetidas


c:\Users\Ignacio\anaconda3\envs\mt5python\Lib\site-packages\pandas\io\parquet.py:190: UserWarning: The DataFrame has column names of mixed type. They will be converted to strings and not roundtrip correctly.
  table = self.api.Table.from_pandas(df, **from_pandas_kwargs)
c:\Users\Ignacio\anaconda3\envs\mt5python\Lib\site-packages\pandas\io\parquet.py:190: UserWarning: The DataFrame has column names of mixed type. They will be converted to strings and not roundtrip correctly.
  table = self.api.Table.from_pandas(df, **from_pandas_kwargs)


AbbVie ✔ No hay ninguna columna con valores nulos
AbbVie ✔ No hay ninguna fila con valores duplicados
AbbVie ✔ No hay fechas repetidas
PepsiCo ✔ No hay ninguna columna con valores nulos
PepsiCo ✔ No hay ninguna fila con valores duplicados
PepsiCo ✔ No hay fechas repetidas
Walmart ✔ No hay ninguna columna con valores nulos
Walmart ✔ No hay ninguna fila con valores duplicados
Walmart ✔ No hay fechas repetidas


c:\Users\Ignacio\anaconda3\envs\mt5python\Lib\site-packages\pandas\io\parquet.py:190: UserWarning: The DataFrame has column names of mixed type. They will be converted to strings and not roundtrip correctly.
  table = self.api.Table.from_pandas(df, **from_pandas_kwargs)
c:\Users\Ignacio\anaconda3\envs\mt5python\Lib\site-packages\pandas\io\parquet.py:190: UserWarning: The DataFrame has column names of mixed type. They will be converted to strings and not roundtrip correctly.
  table = self.api.Table.from_pandas(df, **from_pandas_kwargs)
